In [ ]:
!pip install -q diffusers torch torchvision transformers
!pip install --upgrade accelerate 

## Constants and Paths

In [ ]:
SYNTHETIC_SAVE_DIR = "/kaggle/working/synthetic_bengali_images"
csv_path = "/kaggle/input/sec-translated-bn/secure_translated_bn_valid.csv"
CSV_SAVE_PATH = "/kaggle/working/synthetic_bengali_captioned_images.csv"

## Load Pre-trained Models 

In [ ]:
from diffusers import StableDiffusionPipeline
from transformers import CLIPProcessor, CLIPModel
import torch
import hashlib
from PIL import Image

import torch.nn as nn
import timm
from transformers import AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"

# Build model components
vision_encoder = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=0)
vision_proj = nn.Linear(vision_encoder.num_features, 128)

text_encoder = AutoModel.from_pretrained("sentence-transformers/LaBSE")
text_proj = nn.Linear(768, 128)


# Load the saved state dict
checkpoint = torch.load("/kaggle/input/clip_bn/transformers/default/1/clip_bn.pt", map_location=device)

# Assign weights to each component
vision_encoder.load_state_dict(checkpoint['vision_encoder'])
vision_proj.load_state_dict(checkpoint['vision_proj'])
text_proj.load_state_dict(checkpoint['text_proj'])

# Move to device and set to eval mode
vision_encoder = vision_encoder.to(device).eval()
vision_proj = vision_proj.to(device).eval()
text_encoder = text_encoder.to(device).eval()
text_proj = text_proj.to(device).eval()


In [ ]:
from diffusers import KandinskyPriorPipeline, KandinskyPipeline
import torch
from PIL import Image

# Load models with half precision (saves memory)
prior_pipeline = KandinskyPriorPipeline.from_pretrained(
    "kandinsky-community/kandinsky-2-1-prior", torch_dtype=torch.float16
).to("cuda")

pipeline = KandinskyPipeline.from_pretrained(
    "kandinsky-community/kandinsky-2-1", torch_dtype=torch.float16
).to("cuda")

# Joint prompt: English + Bengali
prompt = "A bullock cart in a village. একটি গ্রামে গরুর গাড়"
negative_prompt = "low quality, bad quality, distortion"

# Get prior embeddings
generator = torch.Generator(device="cuda").manual_seed(42)
image_embeds, negative_image_embeds = prior_pipeline(
    prompt=prompt,
    negative_prompt=negative_prompt,
    guidance_scale=1.0,
    generator=generator
).to_tuple()


image = pipeline(
    prompt=prompt,
    image_embeds=image_embeds,
    negative_prompt=negative_prompt,
    negative_image_embeds=negative_image_embeds,
    height=768,
    width=768,
    num_inference_steps=35,
    generator=generator
).images[0]


image.show()
image.save("joint_prompt_kandinsky21.png")


## Generate Visual Prompts for Bengali Captions

In [ ]:
import torch
from transformers import AutoTokenizer
from torchvision import transforms
from PIL import Image

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load tokenizer for LaBSE
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/LaBSE")

def generate_text_embedding(text, text_encoder, text_proj):
    # Tokenize Bengali text
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

    # Get LaBSE text features
    with torch.no_grad():
        output = text_encoder(**inputs)
        pooled_output = output.last_hidden_state[:, 0]  # CLS token
        text_emb = text_proj(pooled_output)

    # Normalize
    text_emb = text_emb / text_emb.norm(p=2, dim=-1, keepdim=True)
    return text_emb


# Image preprocessing (CLIP style)
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

def generate_image_embedding(image_path, vision_encoder, vision_proj):
    image = Image.open(image_path).convert("RGB")
    img_tensor = image_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        features = vision_encoder(img_tensor)
        image_emb = vision_proj(features)

    # Normalize
    image_emb = image_emb / image_emb.norm(p=2, dim=-1, keepdim=True)
    return image_emb



## Stable Diffusion

In [ ]:

def generate_image_from_caption_kandinsky(caption_bn, prior_pipeline, pipeline, generator):
    image_emb, zero_image_emb = prior_pipeline(
        prompt=caption_bn,
        negative_prompt="low quality, bad anatomy, watermark",
        generator=generator
    ).to_tuple()

    image = pipeline(
        prompt=caption_bn,
        image_embeds=image_emb,
        negative_prompt="low quality, bad anatomy, watermark",
        negative_image_embeds=zero_image_emb,
        height=768,
        width=768,
        num_inference_steps=50,
        generator=generator
    ).images[0]

    return image

## Generate and Save the images 

In [ ]:
import os
import hashlib
import json

def save_image_with_metadata(
    caption,
    generated_image,
    save_dir=SYNTHETIC_SAVE_DIR,
    model_version="kandinsky-2.1"
):
    os.makedirs(save_dir, exist_ok=True)

    # Create a unique metadata hash for this caption-model combo
    metadata_hash = hashlib.sha256(f"{caption}|{model_version}".encode("utf-8")).hexdigest()

    # File paths
    img_filename = f"{metadata_hash}.png"
    img_path = os.path.join(save_dir, img_filename)
    json_path = os.path.join(save_dir, f"{metadata_hash}.json")

    # Metadata dictionary
    metadata = {
        "caption_bn": caption,
        "model_version": model_version,
        "caption_hash": metadata_hash,
        "filename": img_filename
    }

    # Save image
    try:
        generated_image.save(img_path)
    except Exception as e:
        print(f"Error saving image: {e}")
        return None, None

    # Save metadata
    try:
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)
    except Exception as e:
        print(f"Error saving metadata: {e}")

    return img_path, metadata_hash

## Main Image Generation loop

In [ ]:
import pandas as pd

# Load the CSV from Kaggle input directory
df_valid = pd.read_csv(csv_path)

# Display the first few rows to confirm
df_valid.head()


In [ ]:
from tqdm import tqdm

def generate_synthetic_images(df, prior_pipeline, main_pipeline, save_dir=SYNTHETIC_SAVE_DIR):
    generated_paths = []
    metadata_hashes = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        caption_bn = row["caption_bn"]
        caption_en = row["caption_en"]
        joint_prompt = f"{caption_en}. {caption_bn}"

        # Generate image using Kandinsky 2.1 (no ControlNet, no hint)
        generator = torch.Generator(device="cuda").manual_seed(42)
        image_embeds, negative_image_embeds = prior_pipeline(
            prompt=joint_prompt,
            negative_prompt="low quality, bad anatomy, watermark",
            guidance_scale=1.0,
            generator=generator
        ).to_tuple()

        image = main_pipeline(
            prompt=joint_prompt,
            image_embeds=image_embeds,
            negative_prompt="low quality, bad anatomy, watermark",
            negative_image_embeds=negative_image_embeds,
            height=768,
            width=768,
            num_inference_steps=35,
            generator=generator
        ).images[0]

        # Save image with metadata
        img_path, metadata_hash = save_image_with_metadata(
            caption=joint_prompt,
            generated_image=image,
            save_dir=save_dir,
            model_version="kandinsky-2.1"
        )

        generated_paths.append(img_path)
        metadata_hashes.append(metadata_hash)

    df = df.copy()
    df['image_path'] = generated_paths
    df['caption_hash'] = metadata_hashes
    df['model_version'] = 'kandinsky-2.1'

    return df


In [ ]:
# Select 200 entries (index 2500 to 2700)
df_subset = df_valid.iloc[2500:2700]

# Run generation on this slice
df_with_images = generate_synthetic_images(df_subset, prior_pipeline=prior_pipeline, main_pipeline=pipeline)

# Save the result
df_with_images.to_csv(CSV_SAVE_PATH, index=False)
